# Import et test

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import numpy as np
import sys 
from pathlib import Path
import random
from pprint import pp
import time

In [10]:
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.config import CHROMA_DIR,EMBED_MODEL,COLLECTION_NAME,DATA

In [3]:
emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
phrases = [
    "Le ratio CET1 du Groupe s'établit à 12,9 %.",
    "UBS's CET1 capital ratio was 14.4% at year end.",
    "Le Conseil d'administration s'est réuni 9 fois en 2025.",
]
question = "Quel est le ratio CET1 ?"
#question = "Quel est le niveau de solvabilité de la banque ?"
vecteurs = np.array(emb.embed_documents(phrases))
vec_question = np.array(emb.embed_query(question))


print(vecteurs.shape)          # (3, 1024) : 3 phrases, 1 024 nombres chacune
print(vecteurs @ vec_question)     # similarité de chaque phrase avec la question
print(vecteurs @ vecteurs.T)   # similarité de chaque phrase avec les autres

In [ ]:
chunks = read_pages(ROOT / 'data/processed/chunks.jsonl')
extrait_chunks = random.sample(chunks, 50)


# Chroma

In [ ]:
collect_test = Chroma(
    collection_name = 'test_pages',
    embedding_function = emb,
    persist_directory = str(ROOT /'data/chroma_test'),
    collection_configuration = {"hnsw": {"space" : "cosine"}},
)

In [ ]:
collect_test.add_documents(documents=extrait_chunks,
                      ids= [c.metadata['chunk_id'] for c in extrait_chunks])

In [ ]:
collect_test.similarity_search_with_score(query=question, k=3,)

In [ ]:
pp(collect_test.similarity_search_with_score(query=question, k=3,)[0][0].page_content)

In [ ]:
embedding_func = HuggingFaceEmbeddings(model= EMBED_MODEL, encode_kwargs = {"normalize_embeddings": True})


In [6]:
def build_chroma(chunks, taille_lot=500):
    collect = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_func,
        persist_directory = str(CHROMA_DIR),
        collection_configuration = {"hnsw": {"space" : "cosine"}}
    )
    print(f'{len(chunks)} chunks a indexer dans {CHROMA_DIR}')
    for i in range(0,len(chunks), taille_lot):
        chunks_petit = chunks[i:i+taille_lot]
        collect.add_documents(documents=chunks_petit, ids=[c.metadata['chunk_id'] for c in chunks_petit])
        print(f'{i+len(chunks_petit)}/{len(chunks)} chunks ont ete ajoutes dans index Chroma.')
    return collect

In [ ]:
start_time = time.perf_counter()
build_chroma(extrait_chunks)
print(time.perf_counter()-start_time)

In [ ]:
quot,rest = divmod(3567,20)
quot,rest

In [ ]:
20*178

In [10]:
from src.indexing import get_embeddings, build_chroma

In [4]:
chunks = read_pages(ROOT / 'data/processed/chunks.jsonl')

In [11]:
start_time = time.perf_counter()
build_chroma(chunks)
print(time.perf_counter()-start_time)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

5840 chunks a indexer dans /home/yanis/projects/rapFin_assistant/data/index/chroma
500/5840 chunks ont ete ajoutes dans index Chroma.
1000/5840 chunks ont ete ajoutes dans index Chroma.
1500/5840 chunks ont ete ajoutes dans index Chroma.
2000/5840 chunks ont ete ajoutes dans index Chroma.
2500/5840 chunks ont ete ajoutes dans index Chroma.
3000/5840 chunks ont ete ajoutes dans index Chroma.
3500/5840 chunks ont ete ajoutes dans index Chroma.
4000/5840 chunks ont ete ajoutes dans index Chroma.
4500/5840 chunks ont ete ajoutes dans index Chroma.
5000/5840 chunks ont ete ajoutes dans index Chroma.
5500/5840 chunks ont ete ajoutes dans index Chroma.
5840/5840 chunks ont ete ajoutes dans index Chroma.
8332.267400797005


In [17]:
db = load_chroma(collection_name='rapport_v0', out_dir= str(DATA / 'index/chroma_v0'))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [28]:
question = 'Revenu engendre par bnp paribas ?'
source = db.as_retriever(search_kwargs={'k': 3}).invoke(question)[0].page_content
pp(source)

('ACTIVITÉS ET ÉLÉMENTS DE L’ANNÉE 2025 \n'
 '\n'
 'Résultats consolidés de BNP Paribas \n'
 '\n'
 'Le résultat avant impôt du Groupe s’établit ainsi à 17 065 millions d’euros, '
 'en hausse de 5,4 % par rapport à 2024 (16 188 millions d’euros) et le '
 'résultat net part du Groupe à 12 225 millions d’euros (+ 4.6 %/2024), en '
 'ligne avec l’objectif fixé (supérieur à 12 200 millions d’euros). Au 31 '
 'décembre 2025, la rentabilité des fonds propres tangibles non réévalués '
 '(ROTE) est de 11,6 %, en ligne avec l’objectif fixé de 11,5 %. \n'
 '\n'
 'Le Bénéfice Net Par Action s’établit à 10,29 euros, en hausse de 7,5 % par '
 'rapport à 2024. \n'
 '\n'
 'L’actif net comptable par action (1) s’élève à 93,50 euros au 31 décembre '
 '2025. \n'
 '\n'
 '### RÉALISATION DES OBJECTIFS 2025 \n'
 '\n'
 'Les objectifs 2025 au niveau du Groupe sont atteints : \n'
 '\n'
 '■ ROTE : 11,6 % (objectif 2025 : 11,5 %) ; \n'
 '\n'
 '■ Résultat net : 12 225 millions d’euros (objectif 2025 > 12 200 mill

# BM25

In [30]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /home/yanis/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [31]:
import re
import unicodedata

from nltk.corpus import stopwords

In [32]:
def sans_accents(texte):
    texte = unicodedata.normalize("NFKD", texte)
    return "".join(c for c in texte if not unicodedata.combining(c))

MOTS_VIDES = {sans_accents(m) for m in stopwords.words("french") + stopwords.words("english")}

In [34]:
def tokenize(texte):
    texte = re.sub(r"(?<=\d)[ \u00a0\u202f](?=\d{3}\b)", "", texte)    # retire espace dans nombres genre 12 225 millions
    texte = sans_accents(texte.lower())                                # minuscules, sans accents
    texte = re.sub(r"(?<=\d),(?=\d)", ".", texte)                      # remplace , dans nombre decimales en . (conv fr/eng)
    mots = re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", texte)                # cree liste des mots, chiffres, décimales
    return [m for m in mots if len(m) > 1 and m not in MOTS_VIDES]     # sans mots vides ni lettres seules

In [35]:
tokenize('Le résultat avant impôt du Groupe s’établit ainsi à 17 065 millions d’euros, ')

['resultat',
 'avant',
 'impot',
 'groupe',
 'etablit',
 'ainsi',
 '17065',
 'millions',
 'euros']

In [39]:
tokenize(source)

['activites',
 'elements',
 'annee',
 '2025',
 'resultats',
 'consolides',
 'bnp',
 'paribas',
 'resultat',
 'avant',
 'impot',
 'groupe',
 'etablit',
 'ainsi',
 '17065',
 'millions',
 'euros',
 'hausse',
 '5.4',
 'rapport',
 '2024',
 '16188',
 'millions',
 'euros',
 'resultat',
 'net',
 'part',
 'groupe',
 '12225',
 'millions',
 'euros',
 '4.6',
 '2024',
 'ligne',
 'objectif',
 'fixe',
 'superieur',
 '12200',
 'millions',
 'euros',
 '31',
 'decembre',
 '2025',
 'rentabilite',
 'fonds',
 'propres',
 'tangibles',
 'non',
 'reevalues',
 'rote',
 '11.6',
 'ligne',
 'objectif',
 'fixe',
 '11.5',
 'benefice',
 'net',
 'action',
 'etablit',
 '10.29',
 'euros',
 'hausse',
 '7.5',
 'rapport',
 '2024',
 'actif',
 'net',
 'comptable',
 'action',
 'eleve',
 '93.50',
 'euros',
 '31',
 'decembre',
 '2025',
 'realisation',
 'objectifs',
 '2025',
 'objectifs',
 '2025',
 'niveau',
 'groupe',
 'atteints',
 'rote',
 '11.6',
 'objectif',
 '2025',
 '11.5',
 'resultat',
 'net',
 '12225',
 'millions',
 'eur

In [47]:
from rank_bm25 import BM25Okapi

In [69]:
corpus = [tokenize(c.page_content) for c in chunks]
index_bm25 = BM25Okapi(corpus)

In [70]:
scores = index_bm25.get_scores(tokenize(question))
scores

array([0.       , 4.0519757, 0.       , ..., 0.       , 0.       ,
       0.       ], shape=(5840,))

In [84]:
def build_bm25_index(chunks, tokenizer):
    corpus = [ tokenizer(c.page_content) for c in chunks]
    return BM25Okapi(corpus)

In [55]:
np.argsort(-scores)[:5]

array([1440, 1408, 2473, 2519, 1401])

In [63]:
for i in np.argsort(-scores)[:5]:
    print(chunks[i].metadata['chunk_id'], scores[i])

bnpparibas_2025_p369_c2 14.411366724757656
bnpparibas_2025_p364_c5 11.327060549228882
bnpparibas_2025_p660_c1 9.877007148293558
bnpparibas_2025_p672_c0 9.565535535371968
bnpparibas_2025_p363_c7 9.439191448008007


In [87]:
def bm25_search(question:str, chunks, tokenizer, index_bm25:BM25Okapi, k=5):
    """Renvoies liste des k chunks avec le plus grand scores apres une recherche bm25 a la question"""
    scores = index_bm25.get_scores(tokenizer(question))
    top_scores_index = np.argsort(-scores)[:k]
    return [chunks[i] for i in top_scores_index if scores[i]>0]

In [88]:
bm25_search(question,chunks, tokenize, index_bm25)

[Document(metadata={'source': 'bnpparibas_2025.pdf', 'banque': 'BNP Paribas', 'annee': 2025, 'langue': 'fr', 'page': 369, 'chunk_id': 'bnpparibas_2025_p369_c2'}, page_content='Il est également possible que l’application d’exigences renforcées (notamment en matière de capital règlementaire et de limitations d’activité) aux établissements financiers de taille significative ou d’importance systémique, auxquelles les nouveaux acteurs peuvent ne pas être soumis, engendre des distorsions de concurrence préjudiciables aux intérêts des plus grandes institutions du secteur privé, telles que le Groupe BNP Paribas.'),
 Document(metadata={'source': 'bnpparibas_2025.pdf', 'banque': 'BNP Paribas', 'annee': 2025, 'langue': 'fr', 'page': 364, 'chunk_id': 'bnpparibas_2025_p364_c5'}, page_content='- l’augmentation continue des exigences prudentielles prévues notamment par le Règlement du Parlement européen et du Conseil du 26 juin 2013 (tel que modifié ou supplémenté à tout moment, le « CRR ») en matièr

## Creation de retriever BM25Retriever

In [95]:
from typing import Callable
from pydantic import ConfigDict
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi

In [96]:
class BM25Retriever(BaseRetriever):
    """Retriever BM25 maison, compatible LangChain (invoke, EnsembleRetriever…) car langchain-community deprecie."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    chunks: list[Document]
    index: BM25Okapi
    tokenizer: Callable[[str], list[str]]
    k: int = 5

    @classmethod
    def from_documents(cls, chunks, tokenizer, k=5):
        """Construit l'index BM25 à partir des chunks."""
        bm25_index = build_bm25_index(chunks, tokenizer)
        return cls(chunks=chunks,
                   index=bm25_index,
                   tokenizer=tokenizer,k=k
                  )

    def _get_relevant_documents(self, query: str, *,
                                run_manager: CallbackManagerForRetrieverRun) -> list[Document]:
        return bm25_search(question=query,
                           chunks= self.chunks,
                           tokenizer= self.tokenizer,
                           index_bm25= self.index,
                           k=self.k
                          )

In [97]:
bm25_retriever = BM25Retriever.from_documents(chunks,tokenize)

In [99]:
bm25_retriever.invoke(questions[0])

[Document(metadata={'source': 'ubs_2025.pdf', 'banque': 'UBS', 'annee': 2025, 'langue': 'en', 'page': 136, 'chunk_id': 'ubs_2025_p136_c4'}, page_content='The Group Asset and Liability Committee, a committee of the Group Executive Board, has mandated Group Treasury to adjust the currency mix of CET1 capital, within limits set by the BoD, to balance the effect of foreign exchange movements on CET1 capital and the CET1 capital ratio. Limits are in place for the sensitivity of both CET1 capital and the CET1 capital ratio to an appreciation or depreciation of 10% in the value of the US dollar against other currencies.'),
 Document(metadata={'source': 'ubs_2025.pdf', 'banque': 'UBS', 'annee': 2025, 'langue': 'en', 'page': 136, 'chunk_id': 'ubs_2025_p136_c3'}, page_content='1 Includes one debt instrument (ISIN US902613AU26) that ceased to be eligible as gone concern capital when we issued a notice of redemption of the instrument in the fourth quarter of 2025. \n\n###### Additional information

# Fusion BM25/Chroma

In [15]:
from src.chunks import read_pages
from src.bm25 import BM25Retriever,tokenize
from src.chroma import load_chroma
from langchain_classic.retrievers import EnsembleRetriever

In [12]:
chunks = read_pages(DATA / 'processed/chunks.jsonl')

In [16]:
k=10

In [19]:
chroma_retriever = load_chroma().as_retriever(search_kwargs={'k': k})
bm25_retriever = BM25Retriever.from_documents(chunks=chunks, tokenizer=tokenize,k=k)
retrievers = [chroma_retriever,bm25_retriever]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [33]:
def build_hybride_retriever(retrievers, weights=None,c=60,id_key='chunk_id'):
    """Renvoie fusion de retrievers par RRF """
    if weights == None:
        weights = [1/len(retrievers)]*len(retrievers)
    return EnsembleRetriever(retrievers=retrievers,
                                      weights=weights,
                                      c=c,
                                      id_key=id_key)

In [34]:
RRF_retriever = build_hybride_retriever(retrievers,[0.5,0.5])

In [26]:
questions = [
    'Quel est le ratio CET1 ?',
    "Quel est le niveau de solvabilité de la banque ?",
]

In [31]:
RRF_retriever.invoke(questions[0])[0].metadata['chunk_id']

'ubs_2025_p136_c4'

In [32]:
def top_ids(retriever, question, n=5):
    return [d.metadata["chunk_id"] for d in retriever.invoke(question)[:n]]

for q in questions:
    print(q)
    print("  Chroma :", top_ids(chroma_retriever, q))
    print("  BM25   :", top_ids(bm25_retriever, q))
    print("  Hybride:", top_ids(RRF_retriever, q))

Quel est le ratio CET1 ?
  Chroma : ['ubs_2025_p69_c2', 'ubs_2025_p69_c3', 'bnpparibas_2025_p401_c3', 'bnpparibas_2025_p405_c3', 'bnpparibas_2025_p350_c0']
  BM25   : ['ubs_2025_p136_c4', 'ubs_2025_p136_c3', 'ubs_2025_p46_c4', 'ubs_2025_p6_c3', 'ubs_2025_p19_c2']
  Hybride: ['ubs_2025_p136_c4', 'ubs_2025_p19_c1', 'ubs_2025_p69_c2', 'ubs_2025_p69_c3', 'ubs_2025_p136_c3']
Quel est le niveau de solvabilité de la banque ?
  Chroma : ['bnpparibas_2025_p403_c0', 'ubs_2025_p100_c1', 'bnpparibas_2025_p402_c2', 'bnpparibas_2025_p595_c3', 'bnpparibas_2025_p406_c2']
  BM25   : ['bnpparibas_2025_p419_c5', 'bnpparibas_2025_p403_c2', 'bnpparibas_2025_p595_c0', 'ca_2025_p220_c0', 'bnpparibas_2025_p420_c0']
  Hybride: ['bnpparibas_2025_p403_c0', 'bnpparibas_2025_p403_c2', 'bnpparibas_2025_p419_c5', 'ubs_2025_p100_c1', 'bnpparibas_2025_p402_c2']


In [37]:
RRF_retriever.invoke(questions[0])[0].metadata

{'source': 'ubs_2025.pdf',
 'banque': 'UBS',
 'annee': 2025,
 'chunk_id': 'ubs_2025_p136_c4',
 'langue': 'en',
 'page': 136}

# Comparaison index

## Temps creation index
### Index BM25

In [89]:
import time

In [91]:
start = time.perf_counter()
build_bm25_index(chunks,tokenize)
print(f'La creation de l\'index a pris {time.perf_counter() -start}sec.')

La creation de l'index a pris 1.0903803060064092sec.


### Index Chroma

In [ ]:
build_chroma

## Reponses aux questions

In [92]:
questions = [
    'Quel est le ratio CET1 ?',
    "Quel est le niveau de solvabilité de la banque ?",
]

### BM25

In [94]:
for q in questions:
    print(bm25_search(q, tokenizer=tokenize, chunks=chunks, index_bm25=index_bm25))

[Document(metadata={'source': 'ubs_2025.pdf', 'banque': 'UBS', 'annee': 2025, 'langue': 'en', 'page': 136, 'chunk_id': 'ubs_2025_p136_c4'}, page_content='The Group Asset and Liability Committee, a committee of the Group Executive Board, has mandated Group Treasury to adjust the currency mix of CET1 capital, within limits set by the BoD, to balance the effect of foreign exchange movements on CET1 capital and the CET1 capital ratio. Limits are in place for the sensitivity of both CET1 capital and the CET1 capital ratio to an appreciation or depreciation of 10% in the value of the US dollar against other currencies.'), Document(metadata={'source': 'ubs_2025.pdf', 'banque': 'UBS', 'annee': 2025, 'langue': 'en', 'page': 136, 'chunk_id': 'ubs_2025_p136_c3'}, page_content='1 Includes one debt instrument (ISIN US902613AU26) that ceased to be eligible as gone concern capital when we issued a notice of redemption of the instrument in the fourth quarter of 2025. \n\n###### Additional information 